In [ ]:
from pathlib import Path
import sys, os, gc, hashlib, json
code_root=next(p.parent for p in Path('/kaggle/input').rglob('train.json') if (p.parent/'b1_fusion.py').exists())
sys.path.insert(0,str(code_root))
models=next(p.parent for p in Path('/kaggle/input').rglob('b1_latest_model.pt'))
adapters=list(Path('/kaggle/input').rglob('best_adapter.pt'))
assert len(adapters)==1
os.environ['PHASE6_WARM_ADAPTER']=str(adapters[0])
print('Code, parent and trained B1 adapter found',flush=True)


In [ ]:
%%writefile evidence_fusion.py
"""Own-B1 hierarchy + source pointer, adapted from our verified passage fusion.

Original-token memory is preserved. Coverage is a training auxiliary objective;
four-gram blocking is a separate decoding heuristic. Neither ensures factuality.
"""
import hashlib
import math
import os
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
import b1_fusion as base


class EvidenceFusion(base.B1PassageFusion):
    def __init__(self,backbone,rank=8,alpha=16):
        super().__init__(backbone,rank,alpha)
        self.add_evidence_modules()

    def add_evidence_modules(self):
        d=self.cfg.d_model
        self.hierarchy=nn.GRU(d,d//2,batch_first=True,bidirectional=True)
        self.hierarchy_projection=nn.Linear(d,d,bias=False)
        self.hierarchy_gate=nn.Parameter(torch.zeros(()))
        self.pointer_q=nn.Linear(d,min(d,128),bias=False)
        self.pointer_k=nn.Linear(d,min(d,128),bias=False)
        self.pointer_gate=nn.Linear(d,1)
        nn.init.zeros_(self.pointer_gate.weight)
        nn.init.constant_(self.pointer_gate.bias,-4.)
        self.coverage_weight=.05
        self.backbone.decoder_norm.register_forward_hook(self._capture_hidden)

    def _capture_hidden(self,module,inputs,output):self._decoder_hidden=output

    def encode_passages(self,source_ids,source_mask):
        memory,mask=super().encode_passages(source_ids,source_mask)
        b,p,w=source_ids.shape;d=memory.shape[-1]
        tokens=memory.reshape(b,p,w,d)
        valid=source_mask.any(-1)
        pooled=tokens.sum(2)/source_mask.sum(2).clamp_min(1)[...,None]
        # Packing excludes trailing empty passages. Source packer generates only
        # trailing empty padding; reject holes instead of silently mixing them.
        lengths=valid.sum(-1)
        if not torch.equal(valid,torch.arange(p,device=valid.device)[None,:]<lengths[:,None]):
            raise ValueError('Only trailing empty passages supported')
        packed=nn.utils.rnn.pack_padded_sequence(pooled,lengths.cpu(),batch_first=True,enforce_sorted=False)
        contextual,_=self.hierarchy(packed)
        contextual,_=nn.utils.rnn.pad_packed_sequence(contextual,batch_first=True,total_length=p)
        enriched=tokens+torch.tanh(self.hierarchy_gate)*self.hierarchy_projection(contextual)[:,:,None,:]
        enriched=enriched.masked_fill(~source_mask[...,None],0).reshape(b,p*w,d)
        self._encoded_source=(enriched,source_ids.reshape(b,p*w),mask)
        return enriched,mask

    def features(self,source_ids,source_mask,decoder_ids,past_kvs=None,use_cache=False):
        native_cache=None if past_kvs is None else past_kvs['native']
        logits,native=super().forward(source_ids,source_mask,decoder_ids,native_cache,use_cache)
        if past_kvs is None:
            memory,ids,mask=self._encoded_source
            eligible=mask & ids.ge(4)
            keys=self.pointer_k(memory)
        else:
            keys,ids,eligible=past_kvs['keys'],past_kvs['ids'],past_kvs['eligible']
        hidden=self._decoder_hidden
        scores=torch.bmm(self.pointer_q(hidden).float(),keys.float().transpose(1,2))/math.sqrt(keys.shape[-1])
        has=eligible.any(-1)
        safe=eligible | ~has[:,None]
        attention=scores.masked_fill(~safe[:,None],-torch.inf).softmax(-1)*eligible[:,None]
        gate=self.pointer_gate(hidden).float()
        cache={'native':native,'keys':keys,'ids':ids,'eligible':eligible} if use_cache else None
        return logits,attention,gate,ids,has,cache

    def forward(self,source_ids,source_mask,decoder_ids,past_kvs=None,use_cache=False):
        logits,a,g,ids,has,cache=self.features(source_ids,source_mask,decoder_ids,past_kvs,use_cache)
        log_vocab=logits.float().log_softmax(-1)
        copy=torch.zeros_like(log_vocab).scatter_add(-1,ids[:,None].expand(-1,a.shape[1],-1),a)
        mix=torch.logaddexp(F.logsigmoid(-g)+log_vocab,F.logsigmoid(g)+copy.clamp_min(1e-30).log())
        return torch.where(has[:,None,None],mix,log_vocab),cache


def training_loss(model,source_ids,source_mask,target_ids,target_mask):
    labels=target_ids[:,1:]
    valid=target_mask[:,1:].bool()
    if not valid.any():raise ValueError('No supervised target tokens')
    logits,a,g,ids,has,_=model.features(source_ids,source_mask,target_ids[:,:-1])
    # Exact gold-token probability avoids allocating the full pointer vocabulary
    # distribution while training.
    vocab=logits.float().log_softmax(-1).gather(-1,labels[...,None]).squeeze(-1)
    copy=(a*ids[:,None].eq(labels[...,None])).sum(-1)
    mix=torch.logaddexp(F.logsigmoid(-g.squeeze(-1))+vocab,F.logsigmoid(g.squeeze(-1))+copy.clamp_min(1e-30).log())
    mix=torch.where(has[:,None],mix,vocab)
    loss=-mix[valid].mean()
    if model.training:
        masked=a*valid[...,None]
        previous=masked.cumsum(1)-masked
        coverage=torch.minimum(masked,previous).sum(-1)
        loss=loss+model.coverage_weight*(coverage*torch.sigmoid(g.squeeze(-1)))[valid].mean()
    return loss


@torch.inference_mode()
def generate(model,source_ids,source_mask,max_new_tokens,eos_id):
    was_training=model.training;model.eval()
    try:
        rows=[[] for _ in range(source_ids.shape[0])]
        token=torch.full((len(rows),1),model.bos_id,device=source_ids.device,dtype=torch.long)
        done=torch.zeros(len(rows),dtype=torch.bool,device=source_ids.device);cache=None
        for step in range(max_new_tokens):
            logp,cache=model(source_ids if cache is None else None,source_mask if cache is None else None,token,past_kvs=cache,use_cache=True)
            scores=logp[:,-1].clone()
            for i,row in enumerate(rows):
                if done[i]:continue
                if step<min(64,max_new_tokens//2):scores[i,eos_id]=-torch.inf
                if len(row)>=3:
                    prefix=tuple(row[-3:])
                    blocked={row[j+3] for j in range(len(row)-3) if tuple(row[j:j+3])==prefix}
                    if blocked:scores[i,list(blocked)]=-torch.inf
                if not torch.isfinite(scores[i]).any():scores[i,eos_id]=0.
            chosen=scores.argmax(-1)
            for i,value in enumerate(chosen.tolist()):
                if not done[i]:rows[i].append(value)
            done |= chosen.eq(eos_id)
            if done.all():break
            token=chosen.masked_fill(done,eos_id)[:,None]
        return rows
    finally:model.train(was_training)


def build(input_dir,device):
    model,tokenizer,metadata=base.build(input_dir,'cpu')
    warm=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path(input_dir)
    candidates=[Path(os.environ['PHASE6_WARM_ADAPTER'])] if os.environ.get('PHASE6_WARM_ADAPTER') else list(warm.rglob('best_adapter.pt'))
    if len(candidates)!=1:raise ValueError('Exactly one B1 Phase 6 adapter source required')
    path=candidates[0];record=torch.load(path,map_location='cpu',weights_only=False)
    if record['parent_metadata']['parent_checkpoint_sha256']!=metadata['parent_checkpoint_sha256']:
        raise ValueError('Warm-start parent mismatch')
    if record['parent_metadata']['tokenizer_sha256']!=metadata['tokenizer_sha256']:
        raise ValueError('Warm-start tokenizer mismatch')
    params=dict(model.named_parameters())
    expected={n for n,p in params.items() if p.requires_grad}
    if set(record['adapter'])!=expected:raise ValueError('Warm-start adapter parameters mismatch')
    with torch.no_grad():
        for n,value in record['adapter'].items():params[n].copy_(value)
    model.__class__=EvidenceFusion
    model.add_evidence_modules()
    model=model.to(device)
    metadata.update({'method':'B1 hierarchical token evidence fusion with pointer generator and trained coverage',
        'warm_start_adapter_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'warm_start_epoch':record['epoch'],
        'hierarchy':'masked passage means, bidirectional GRU, zero-gated residual into original token memory',
        'coverage_weight':model.coverage_weight,'decoding':'greedy, four-gram blocking, minimum64 tokens subject to budget',
        'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
        'total_parameters':sum(p.numel() for p in model.parameters())})
    return model,tokenizer,metadata


In [ ]:
%%writefile train_evidence.py
"""Resumable Phase 6 generation training from the student's Phase 1 weights."""
import argparse
import contextlib
import hashlib
import importlib
import json
import math
import random
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from phase6_data import normalize_rows, split_rows, select_passages, validate_predictions, document_hash


def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()


def save_json(path,value):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(value,indent=2),encoding='utf-8')
    tmp.replace(path)


def trainable_state(model):
    return {name:p.detach().cpu() for name,p in model.named_parameters() if p.requires_grad}


def restore_trainable(model,state):
    params=dict(model.named_parameters())
    with torch.no_grad():
        for name,value in state.items():
            if name not in params or not params[name].requires_grad:
                raise ValueError('Adapter state mismatch: '+name)
            params[name].copy_(value.to(params[name].device))


def pack_source(tokens,method,tokenizer,passages,width,device):
    parts=select_passages(tokens,passages,width)
    if method in ('b1_fusion','b1_adalora','evidence_fusion'):
        ids=torch.full((1,len(parts),max(map(len,parts))),tokenizer.pad_id,dtype=torch.long,device=device)
        mask=torch.zeros_like(ids,dtype=torch.bool)
        for j,part in enumerate(parts):
            ids[0,j,:len(part)]=torch.tensor(part,device=device)
            mask[0,j,:len(part)]=True
    else:
        flat=[token for part in parts for token in part]
        ids=torch.tensor([flat],dtype=torch.long,device=device)
        mask=torch.ones_like(ids,dtype=torch.bool)
    return ids,mask


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--input',required=True)
    parser.add_argument('--models-input',default=None)
    parser.add_argument('--output',required=True)
    parser.add_argument('--method',choices=['evidence_fusion','b1_fusion','legalspan_lora','b1_adalora','c1_houlsby','d222_prefix','a0_kd'],required=True)
    parser.add_argument('--epochs',type=int,default=6)
    parser.add_argument('--accum',type=int,default=8)
    parser.add_argument('--lr',type=float,default=0.00015)
    parser.add_argument('--passages',type=int,default=8)
    parser.add_argument('--passage-tokens',type=int,default=384)
    parser.add_argument('--target-tokens',type=int,default=1536)
    parser.add_argument('--validation-limit',type=int,default=32)
    parser.add_argument('--confirmation-limit',type=int,default=64)
    parser.add_argument('--max-train-hours',type=float,default=5.0)
    parser.add_argument('--resume',default=None)
    parser.add_argument('--teacher-predictions',default=None)
    parser.add_argument('--skip-bertscore',action='store_true')
    args=parser.parse_args()
    if min(args.epochs,args.accum,args.passages,args.passage_tokens,args.target_tokens)<1:
        raise ValueError('Training dimensions must be positive')
    random.seed(781)
    np.random.seed(781)
    torch.manual_seed(781)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(781)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    inp,out=Path(args.input),Path(args.output)
    out.mkdir(parents=True,exist_ok=True)
    train_path=next(inp.rglob('train.json'))
    test_path=next(inp.rglob('test.json'))
    raw_train=json.loads(train_path.read_text(encoding='utf-8-sig'))
    train=normalize_rows(raw_train,True,drop_empty_summaries=True)
    excluded_empty_summaries=len(raw_train)-len(train)
    test=normalize_rows(json.loads(test_path.read_text(encoding='utf-8-sig')),False)
    test_hashes={document_hash(row['document']) for row in test}
    eligible=[row for row in train if document_hash(row['document']) not in test_hashes]
    splits=split_rows(eligible)
    if not all(splits.values()): raise ValueError('One grouped split is empty')
    teacher_rows=[]
    teacher_sha256=None
    if args.method=='a0_kd':
        if not args.teacher_predictions:
            raise ValueError('A0 knowledge distillation requires teacher predictions')
        teacher_path=Path(args.teacher_predictions)
        teacher_sha256=sha(teacher_path)
        teacher_map=json.loads(teacher_path.read_text(encoding='utf-8'))
        fit_by_id={row['id']:row for row in splits['fit']}
        if (not isinstance(teacher_map,dict) or len(teacher_map)<64
                or not set(teacher_map).issubset(fit_by_id)
                or any(not isinstance(summary,str) or not summary.strip()
                       for summary in teacher_map.values())):
            raise ValueError('Teacher targets must be nonempty and fitting-only')
        # Three teacher passes give the 256 generated targets a meaningful
        # fraction of updates while retaining every gold fitting example.
        teacher_rows=[dict(fit_by_id[item],summary=summary)
                      for item,summary in sorted(teacher_map.items()) for _ in range(3)]
    elif args.teacher_predictions:
        raise ValueError('Teacher predictions are reserved for the KD experiment')
    split_manifest={'data_sha256':{'train':sha(train_path),'test':sha(test_path)},'ids':{k:[r['id'] for r in v] for k,v in splits.items()},'excluded_empty_summaries':excluded_empty_summaries,'excluded_train_test_exact_duplicates':len(train)-len(eligible)}
    identity_args={k:v for k,v in vars(args).items() if k not in ('input','models_input','output','resume','max_train_hours','teacher_predictions')}
    run_identity={'split':split_manifest,'settings':identity_args,'teacher_predictions_sha256':teacher_sha256}
    save_json(out/'split_manifest.json',split_manifest)
    module=importlib.import_module(args.method)
    models_input=Path(args.models_input) if args.models_input else inp
    source_archive=models_input/'legalspan_source.zip'
    if args.method=='legalspan_lora' and source_archive.exists():
        runtime_source=out/'runtime_legalspan_source'
        with zipfile.ZipFile(source_archive) as archive:
            for name in archive.namelist():
                resolved=(runtime_source/name).resolve()
                if not resolved.is_relative_to(runtime_source.resolve()):
                    raise ValueError('Unsafe source archive path')
            archive.extractall(runtime_source)
        sys.path.insert(0,str(runtime_source))
    model,tokenizer,metadata=module.build(models_input,device)
    if teacher_sha256 is not None:
        metadata['teacher_predictions_sha256']=teacher_sha256
        metadata['teacher_fit_documents']=len(teacher_map)
    print(json.dumps({'event':'model_loaded','device':device,'metadata':metadata}),flush=True)
    if device!='cuda': raise RuntimeError('GPU training requested, but CUDA is unavailable')
    scorer=None
    if not args.skip_bertscore:
        from bert_score import BERTScorer
        scorer=BERTScorer(lang='en',device=device,batch_size=8)
    def encode_rows(rows):
        result=[]
        for i,row in enumerate(rows):
            source=tokenizer.encode(row['document'])
            target=tokenizer.encode(row['summary'])[:args.target_tokens-1] if 'summary' in row else None
            result.append((row,source,target))
            if (i+1)%500==0: print(json.dumps({'event':'tokenized','rows':i+1}),flush=True)
        return result
    encoded={k:encode_rows(v) for k,v in splits.items()}
    if teacher_rows:
        encoded['fit'].extend(encode_rows(teacher_rows))
    encoded_test=encode_rows(test)
    new_prefixes=('hierarchy','pointer_')
    new_params=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith(new_prefixes)]
    old_params=[p for n,p in model.named_parameters() if p.requires_grad and not n.startswith(new_prefixes)]
    optimizer=torch.optim.AdamW([{'params':old_params,'lr':args.lr},{'params':new_params,'lr':args.lr*3}],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda')
    history=[]
    best_score=-math.inf
    start_epoch=start_offset=updates=0
    if args.resume:
        record=torch.load(args.resume,map_location='cpu',weights_only=False)
        if record['run_identity']!=run_identity: raise ValueError('Resume data, split or training settings differ')
        old_hashes={k:v for k,v in record['parent_metadata'].items() if 'sha256' in k}
        new_hashes={k:v for k,v in metadata.items() if 'sha256' in k}
        if old_hashes!=new_hashes: raise ValueError('Resume parent provenance differs')
        restore_trainable(model,record['adapter'])
        optimizer.load_state_dict(record['optimizer'])
        scaler.load_state_dict(record['scaler'])
        history=record['history']
        best_score=record['best_score']
        start_epoch,start_offset,updates=record['epoch'],record['next_offset'],record['updates']
        random.setstate(record['random_state'])
        np.random.set_state(record['numpy_state'])
        torch.set_rng_state(record['torch_state'])
        torch.cuda.set_rng_state_all(record['cuda_states'])
        if record.get('best_record') is not None:
            torch.save(record['best_record'],out/'best_adapter.pt')
    def checkpoint(epoch,next_offset):
        best_record=torch.load(out/'best_adapter.pt',map_location='cpu',weights_only=False) if (out/'best_adapter.pt').exists() else None
        record={'adapter':trainable_state(model),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'parent_metadata':metadata,'run_identity':run_identity,'best_record':best_record,'epoch':epoch,'next_offset':next_offset,'updates':updates,'history':history,'best_score':best_score,'random_state':random.getstate(),'numpy_state':np.random.get_state(),'torch_state':torch.get_rng_state(),'cuda_states':torch.cuda.get_rng_state_all(),'args':vars(args)}
        tmp=out/'latest.tmp'
        torch.save(record,tmp)
        tmp.replace(out/'latest.pt')
    @torch.no_grad()
    def generate_rows(rows):
        model.eval()
        predictions=[]
        for generation_index,(row,source,_) in enumerate(rows):
            print(json.dumps({'event':'generation_start','index':generation_index+1,'count':len(rows),'id':row['id']}),flush=True)
            ids,mask=pack_source(source,args.method,tokenizer,args.passages,args.passage_tokens,device)
            with torch.autocast('cuda',dtype=torch.float16):
                result=module.generate(model,ids,mask,args.target_tokens,tokenizer.eos_id)[0]
            cleaned=[]
            for token in result:
                if token==tokenizer.eos_id: break
                if token not in (tokenizer.pad_id,tokenizer.bos_id): cleaned.append(int(token))
            text=tokenizer.decode(cleaned).strip()
            if not text: raise RuntimeError('Empty generated summary: '+row['id'])
            predictions.append({'id':row['id'],'summary':text})
            print(json.dumps({'event':'generation_complete','index':generation_index+1,'count':len(rows),'id':row['id'],'words':len(text.split())}),flush=True)
        return predictions
    def evaluate(name,limit):
        subset=sorted(encoded[name],key=lambda item:hashlib.sha256(item[0]['id'].encode()).hexdigest())[:limit]
        predictions=generate_rows(subset)
        refs=[row['summary'] for row,_,_ in subset]
        save_json(out/(name+'_predictions.json'),predictions)
        if scorer:
            _,_,scores=scorer.score([r['summary'] for r in predictions],refs,batch_size=8)
            result={'bertscore_f1':float(scores.mean()),'count':len(subset),'scorer_hash':scorer.hash}
        else:
            result={'bertscore_f1':None,'count':len(subset),'selection':'negative_summary_nll'}
            losses=[]
            model.eval()
            with torch.no_grad():
                for row,source,target in subset:
                    ids,mask=pack_source(source,args.method,tokenizer,args.passages,args.passage_tokens,device)
                    targets=torch.tensor([[tokenizer.bos_id]+target+[tokenizer.eos_id]],device=device)
                    with torch.autocast('cuda',dtype=torch.float16):
                        loss=module.training_loss(model,ids,mask,targets,torch.ones_like(targets,dtype=torch.bool))
                    losses.append(float(loss))
            result['negative_nll']=-float(np.mean(losses))
        return result
    began=time.monotonic()
    stop=False
    for epoch in range(start_epoch,args.epochs):
        order=list(range(len(encoded['fit'])))
        random.Random(781+epoch).shuffle(order)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running=[]
        offset=start_offset if epoch==start_epoch else 0
        for pos in range(offset,len(order)):
            row,source,target=encoded['fit'][order[pos]]
            ids,mask=pack_source(source,args.method,tokenizer,args.passages,args.passage_tokens,device)
            targets=torch.tensor([[tokenizer.bos_id]+target+[tokenizer.eos_id]],device=device)
            group_start=(pos//args.accum)*args.accum
            group_size=min(args.accum,len(order)-group_start)
            with torch.autocast('cuda',dtype=torch.float16):
                loss=module.training_loss(model,ids,mask,targets,torch.ones_like(targets,dtype=torch.bool))
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite training loss')
            scaler.scale(loss/group_size).backward()
            running.append(float(loss.detach()))
            if (pos+1)%args.accum==0 or pos+1==len(order):
                scaler.unscale_(optimizer)
                trainable=[p for p in model.parameters() if p.requires_grad]
                gradients_finite=all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
                if gradients_finite:
                    torch.nn.utils.clip_grad_norm_(trainable,1.0)
                else:
                    print(json.dumps({'event':'amp_overflow_skipped','epoch':epoch+1,'row':pos+1,'scale':scaler.get_scale()}),flush=True)
                scaler.step(optimizer)
                scaler.update()
                if gradients_finite and hasattr(module,'on_optimizer_step'):
                    module.on_optimizer_step(model,updates+1)
                optimizer.zero_grad(set_to_none=True)
                updates+=int(gradients_finite)
                if updates<=3 or updates%20==0:
                    print(json.dumps({'event':'training','epoch':epoch+1,'row':pos+1,'rows':len(order),'updates':updates,'mean_loss':float(np.mean(running[-160:])),'elapsed_minutes':(time.monotonic()-began)/60}),flush=True)
                if updates%100==0: checkpoint(epoch,pos+1)
                if time.monotonic()-began>args.max_train_hours*3600:
                    checkpoint(epoch,pos+1)
                    stop=True
                    break
        val=evaluate('validation',args.validation_limit)
        metric=val['bertscore_f1'] if val['bertscore_f1'] is not None else val['negative_nll']
        history.append({'epoch':epoch+1,'updates':updates,'train_objective_nll_plus_coverage':float(np.mean(running)) if running else None,'validation':val,'partial_epoch':stop})
        print(json.dumps(history[-1]),flush=True)
        save_json(out/'training_history.json',history)
        epoch_record={'adapter':trainable_state(model),'parent_metadata':metadata,'epoch':epoch+1,'updates':updates,'validation':val,'args':vars(args)}
        torch.save(epoch_record,out/('epoch_'+str(epoch+1)+'_adapter.pt'))
        save_json(out/('epoch_'+str(epoch+1)+'_validation_predictions.json'),json.loads((out/'validation_predictions.json').read_text()))
        if metric>best_score:
            best_score=metric
            torch.save({'adapter':trainable_state(model),'parent_metadata':metadata,'epoch':epoch+1,'updates':updates,'validation':val,'args':vars(args)},out/'best_adapter.pt')
            save_json(out/'best_validation_predictions.json',json.loads((out/'validation_predictions.json').read_text()))
        if not stop: checkpoint(epoch+1,0)
        if stop: break
    best=torch.load(out/'best_adapter.pt',map_location='cpu',weights_only=False)
    restore_trainable(model,best['adapter'])
    confirmation=evaluate('confirmation',args.confirmation_limit)
    predictions=generate_rows(encoded_test)
    validate_predictions(predictions,[r['id'] for r in test])
    zip_path=out/('phase6_'+args.method+'_prediction.zip')
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('prediction.json',json.dumps(predictions,ensure_ascii=False,separators=(',',':')))
    report={'method':args.method,'phase1_parent':metadata,'counts':{k:len(v) for k,v in splits.items()},'test_count':len(test),'selected_epoch':best['epoch'],'validation':best['validation'],'confirmation':confirmation,'passage_strategy':'document-only uniform coverage including first and last; no reference-dependent passage selection','args':vars(args),'zip_sha256':sha(zip_path),'elapsed_minutes':(time.monotonic()-began)/60,'codabench_score':None}
    save_json(out/'report.json',report)
    print(json.dumps({'event':'complete','report':report}),flush=True)


if __name__=='__main__': main()


In [ ]:
import torch, evidence_fusion
assert torch.cuda.is_available()
probe_model,probe_tok,probe_meta=evidence_fusion.build(models,'cuda')
s=torch.tensor([[probe_tok.encode('The court allowed the appeal.')]],device='cuda')
t=torch.tensor([[probe_tok.bos_id]+probe_tok.encode('The appeal was allowed.')+[probe_tok.eos_id]],device='cuda')
with torch.autocast('cuda',dtype=torch.float16):
    probe_loss=evidence_fusion.training_loss(probe_model,s,s.ne(0),t,t.ne(0))
probe_loss.backward()
assert torch.isfinite(probe_loss)
assert probe_model.pointer_gate.weight.grad is not None
assert all(p.grad is None for p in probe_model.parameters() if not p.requires_grad)
print(json.dumps({'event':'cuda_backward_probe_passed','loss':float(probe_loss.detach()),'parent':probe_meta['parent_checkpoint_sha256'],'warm_start_epoch':probe_meta['warm_start_epoch'],'trainable':probe_meta['trainable_parameters']}),flush=True)
del probe_loss,probe_model,s,t
# Backbone hook and temporary model references may form cycles.
gc.collect();torch.cuda.empty_cache()


In [ ]:
import train_evidence
sys.argv=['train_evidence.py','--input',str(code_root),'--models-input',str(models),'--output','/kaggle/working/evidence_fusion','--method','evidence_fusion','--epochs','6','--accum','8','--lr','0.00005','--passages','16','--passage-tokens','256','--target-tokens','768','--validation-limit','16','--confirmation-limit','32','--max-train-hours','3','--skip-bertscore']
train_evidence.main()
